# 03 — Evaluation Debug
Runs all 4 evaluators on all 9 non-DP synthetic datasets.
Produces baseline utility and privacy scores before DP training.

In [ ]:
import os
os.chdir('..')

import json
import yaml
import pandas as pd
import numpy as np
from pathlib import Path

with open('config.yaml') as f:
    config = yaml.safe_load(f)

from src.evaluation.utility.statistical import StatisticalEvaluator
from src.evaluation.utility.tstr import TSTREvaluator
from src.evaluation.privacy.mia import MIAEvaluator
from src.evaluation.privacy.distance_metrics import DistanceMetricsEvaluator

stat_eval  = StatisticalEvaluator(config)
tstr_eval  = TSTREvaluator(config)
mia_eval   = MIAEvaluator(config)
dist_eval  = DistanceMetricsEvaluator(config)

DATASETS   = ['diabetes_130us', 'home_credit', 'acs_income']
GENERATORS = ['ctgan', 'tvae', 'copulagan']

print('Evaluators loaded.')

In [ ]:
def load_dataset(config, dataset_name):
    """Load real train/test CSVs and restore categorical dtypes from meta.json."""
    processed_dir = Path(config['datasets'][dataset_name]['processed_dir'])
    train_df = pd.read_csv(processed_dir / 'train.csv')
    test_df  = pd.read_csv(processed_dir / 'test.csv')

    with open(processed_dir / 'meta.json') as f:
        meta = json.load(f)

    for col, col_meta in meta.get('columns', {}).items():
        if col_meta['type'] == 'categorical':
            if col in train_df.columns:
                train_df[col] = train_df[col].astype(str)
                test_df[col]  = test_df[col].astype(str)

    return train_df, test_df, meta


def load_synthetic(config, dataset_name, label):
    """Load a synthetic CSV."""
    synth_path = Path(config['datasets'][dataset_name]['synthetic_dir']) / f'{label}.csv'
    if not synth_path.exists():
        raise FileNotFoundError(f'Synthetic file not found: {synth_path}')
    return pd.read_csv(synth_path)


print('Helper functions defined.')

In [ ]:
all_results = []

for dataset_name in DATASETS:
    print(f'\n{"="*60}')
    print(f'  Dataset: {dataset_name}')
    print(f'{"="*60}')

    target_col = config['datasets'][dataset_name]['target_col']
    real_train, real_test, meta = load_dataset(config, dataset_name)

    for gen_name in GENERATORS:
        label = f'{gen_name}_nodp'
        print(f'\n  [{label}]')

        try:
            synthetic = load_synthetic(config, dataset_name, label)

            # Align synthetic dtypes with real
            for col in synthetic.columns:
                if col in real_train.columns:
                    if real_train[col].dtype == object:
                        synthetic[col] = synthetic[col].astype(str)

            # 1. Statistical
            stat_res  = stat_eval.evaluate(real_train, synthetic, dataset_name, label)

            # 2. TSTR
            tstr_res  = tstr_eval.evaluate(real_train, real_test, synthetic, target_col, dataset_name, label)

            # 3. MIA
            mia_res   = mia_eval.evaluate(real_train, real_test, synthetic, target_col, dataset_name, label)

            # 4. Distance metrics
            dist_res  = dist_eval.evaluate(real_train, real_test, synthetic, target_col, dataset_name, label)

            # Collect into flat row
            row = {
                'dataset':           dataset_name,
                'generator':         gen_name,
                'epsilon':           'nodp',
                'label':             label,
                # Utility
                'stat_overall':      stat_res['overall_score'],
                'wasserstein_mean':  stat_res['wasserstein_mean'],
                'corr_diff':         stat_res['correlation_matrix_diff'],
                'cat_similarity':    stat_res['categorical_similarity_mean'],
                'tstr_auc':          tstr_res['tstr']['roc_auc'],
                'trtr_auc':          tstr_res['trtr']['roc_auc'],
                'tstr_f1':           tstr_res['tstr']['f1_weighted'],
                'tstr_accuracy':     tstr_res['tstr']['accuracy'],
                'utility_ratio':     tstr_res['utility_ratio'],
                # Privacy
                'mia_auc':           mia_res['mia_auc'],
                'mia_privacy_score': mia_res['privacy_score'],
                'dcr_mean':          dist_res['dcr_mean'],
                'dcr_median':        dist_res['dcr_median'],
                'nndr_mean':         dist_res['nndr_mean'],
                'nndr_median':       dist_res['nndr_median'],
                'dist_privacy_score': dist_res['privacy_score'],
            }
            all_results.append(row)
            print(f'    TSTR AUC: {row["tstr_auc"]:.4f} | MIA AUC: {row["mia_auc"]:.4f} | DCR: {row["dcr_mean"]:.4f}')

        except Exception as e:
            print(f'    ERROR: {e}')

print('\nAll evaluations complete.')

In [ ]:
# Summary table
results_df = pd.DataFrame(all_results)
display_cols = ['dataset','generator','tstr_auc','trtr_auc','utility_ratio','mia_auc','dcr_mean','nndr_median']
print(results_df[display_cols].to_string(index=False))

# Save
Path('outputs/results').mkdir(parents=True, exist_ok=True)
results_df.to_csv('outputs/results/baseline_scores.csv', index=False)
print('\nSaved to outputs/results/baseline_scores.csv')